In [54]:
import pandas as pd

marketing = pd.read_csv("data/raw/marketing_touches.csv")
surveys = pd.read_csv("data/raw/nps_surveys.csv")

In [55]:
marketing.head()

,touch_id,org_id,campaign,channel,timestamp,clicked,converted
0,mkt_7322hr5s,org_zbikcidk,security_week,ads,2025-08-26,False,False
1,mkt_osc0whs1,org_5iqvnb4g,webinar_finops,email,2025-07-23,False,False
2,mkt_4eci9rtj,org_okep7y6w,security_week,email,2025-06-21,False,True
3,mkt_6u5a2x54,org_ohp6vz41,security_week,in_app,2025-08-09,True,False
4,mkt_u44qunf5,org_c11ertj5,upgrade_enterprise,email,2025-07-28,True,False


In [56]:
marketing.shape

(1500, 7)

In [57]:
marketing.dtypes

,0
touch_id,object
org_id,object
campaign,object
channel,object
timestamp,object
clicked,bool
converted,bool


In [58]:
marketing.columns


Index(['touch_id', 'org_id', 'campaign', 'channel', 'timestamp', 'clicked',
       'converted'],
      dtype='object')

In [59]:
marketing.isnull().sum()

,0
touch_id,0
org_id,0
campaign,0
channel,0
timestamp,0
clicked,0
converted,0


In [60]:
marketing.duplicated().sum()

np.int64(0)

In [61]:
marketing["touch_id"].duplicated().sum()

np.int64(0)

In [62]:
marketing["campaign"].value_counts()

,count
campaign,
upgrade_enterprise,271
webinar_finops,263
genai_launch,252
welcome,247
upgrade_pro,245
security_week,222


In [63]:
marketing["channel"].value_counts()

,count
channel,
event,401
email,385
ads,358
in_app,356


In [64]:
marketing[(marketing["clicked"] == False) & (marketing["converted"] == True)]

,touch_id,org_id,campaign,channel,timestamp,clicked,converted
2,mkt_4eci9rtj,org_okep7y6w,security_week,email,2025-06-21,False,True
36,mkt_5uhgcrkf,org_i7p5tb94,upgrade_enterprise,in_app,2025-08-14,False,True
75,mkt_tscjlei6,org_0lvsnujz,upgrade_enterprise,ads,2025-08-07,False,True
90,mkt_wr3xuxnb,org_fel6246h,welcome,event,2025-06-28,False,True
123,mkt_tdl35jrj,org_ohp6vz41,welcome,in_app,2025-06-26,False,True
...,...,...,...,...,...,...,...
1449,mkt_66wqx1n6,org_9t84azyt,security_week,event,2025-08-17,False,True
1468,mkt_r3urev2z,org_ly8ozcyw,upgrade_pro,ads,2025-06-14,False,True
1474,mkt_botm1cd4,org_253j2d54,genai_launch,in_app,2025-07-15,False,True
1488,mkt_nh83fi4o,org_8mdd4v30,genai_launch,event,2025-07-19,False,True


In [65]:
marketing["timestamp"] = pd.to_datetime(marketing["timestamp"])

In [66]:
marketing.dtypes

,0
touch_id,object
org_id,object
campaign,object
channel,object
timestamp,datetime64[ns]
clicked,bool
converted,bool


In [67]:
marketing["timestamp"].isnull().sum()

np.int64(0)

In [68]:
marketing["org_id"].nunique()

80

In [69]:
#las 1.500 interacciones corresponden a 80 organizaciones diferentes

In [70]:
marketing.groupby("org_id").size().describe()

,0
count,80.000000
mean,18.750000
std,4.604621
min,11.000000
25%,15.750000
50%,18.000000
75%,21.250000
max,32.000000


In [71]:
marketing["mes"] = marketing["timestamp"].dt.to_period("M")

marketing["mes"].value_counts().sort_index()

,count
mes,
2025-05,330
2025-06,373
2025-07,415
2025-08,382


In [72]:
surveys.head()

,org_id,survey_date,nps_score,comment
0,org_xaji0y6d,2025-08-22,11.0,Complex billing
1,org_hv3a3zmf,2025-06-13,6.0,Complex billing
2,org_hv3a3zmf,2025-06-18,15.0,Love genAI features
3,org_8mdd4v30,2025-05-24,19.0,Missing features
4,org_8mdd4v30,2025-08-26,47.0,Love genAI features


In [73]:
surveys.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   org_id       92 non-null     object 
 1   survey_date  92 non-null     object 
 2   nps_score    73 non-null     float64
 3   comment      82 non-null     object 
dtypes: float64(1), object(3)
memory usage: 3.0+ KB


In [74]:
# transfprmamos la fecha a formato datetime
surveys["survey_date"] = pd.to_datetime(
    surveys["survey_date"],
    errors="coerce"
)

In [75]:
#verificamos
surveys.dtypes

,0
org_id,object
survey_date,datetime64[ns]
nps_score,float64
comment,object


In [76]:
surveys.shape

(92, 4)

In [77]:
# 92 encuestas de 60 organizaciones diferentes, por lo que algunas organizaciones tienen más de una encuesta

In [78]:
surveys.isnull().sum()

,0
org_id,0
survey_date,0
nps_score,19
comment,10


In [79]:
pd.crosstab(
    surveys["nps_score"].isnull(),
    surveys["comment"].isnull()
)

comment,False,True
nps_score,,
False,64,9
True,18,1


In [80]:
# hay 19 encuestas sin NPS, 18 tienen comentario
# hay 9 encuestas sin comentario pero con NPS, 8 tienen NPS
# hay una encuentas sin comentario y sin Nps

In [81]:
# elq ue no tiene ni nps ni comentario no aporta nada
surveys[
    surveys["nps_score"].isnull() & surveys["comment"].isnull()
]

,org_id,survey_date,nps_score,comment
8,org_5lxo6qji,2025-06-07,NaN,NaN


In [82]:
surveys_clean = surveys.dropna(
    subset=["nps_score", "comment"],
    how="all"
)

In [83]:
surveys_clean[
    surveys_clean["nps_score"].isnull() & surveys_clean["comment"].isnull()
]

,org_id,survey_date,nps_score,comment


In [84]:
surveys["comment"].value_counts(dropna=False)

,count
comment,
Missing features,20
Love genAI features,14
Complex billing,13
Stable but slow,13
Too expensive,12
NaN,10
Great support,10


In [85]:
# tiene 6 categorías de texto predefinidas y 10 valores nulos (categorias aparentemente consistentes)

In [86]:
surveys.duplicated().sum()

np.int64(0)

In [87]:
#sin duplicados

In [88]:
surveys["org_id"].value_counts()

,count
org_id,
org_hv3a3zmf,2
org_8mdd4v30,2
org_9t84azyt,2
org_ujv6oh9s,2
org_53lc58dr,2
org_chj755nf,2
org_pac56t4u,2
org_n9j2qp89,2
org_dppq0y9d,2


In [89]:
# organizaciones con mas de una encuesta, 60 organizaciones, 92 encuestas

In [90]:
surveys.duplicated(
    subset=["org_id", "survey_date"]
).sum()

np.int64(0)

In [91]:
# no existen dos encuestas de la misma organizacion en la misma fecha
# podemos pensar que: org_id + survey_date, podria funcionar como clave natural de la encuesta

In [92]:
surveys["nps_score"].describe()

,nps_score
count,73.000000
mean,26.164384
std,15.893756
min,-16.000000
25%,17.000000
50%,25.000000
75%,36.000000
max,68.000000


In [93]:
# valor min -16 y mas 68, podriamos pensar que los valores van del -100 al 100, a corroborar

In [94]:
q1 = surveys["nps_score"].quantile(0.25)
q3 = surveys["nps_score"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Límite inferior:", lower)
print("Límite superior:", upper)

Q1: 17.0
Q3: 36.0
IQR: 19.0
Límite inferior: -11.5
Límite superior: 64.5


In [95]:
# segun este criterio los valoresmenores a -11,5 o mayor a 64,5 sería considerado un outlier, esto no quiere decir que el dato sea invalido, pero nos ayuda a tener una imagen de los resultaados de la encuesta


In [96]:
surveys["nps_score"].value_counts().sort_index()

,count
nps_score,
-16.0,1
-6.0,1
-5.0,1
0.0,2
5.0,1
6.0,1
9.0,3
11.0,2
12.0,1


In [97]:
print("Fecha mínima:", surveys["survey_date"].min())
print("Fecha máxima:", surveys["survey_date"].max())

Fecha mínima: 2025-05-24 00:00:00
Fecha máxima: 2025-08-31 00:00:00


In [98]:
surveys["survey_date"].nunique()

57

In [99]:
# esta info de las fechas peude ser luego util para analizar la evolucion de satisfaccion

In [100]:
surveys.groupby("comment")["nps_score"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
comment,,,,,
Complex billing,11,19.454545,19.0,-16.0,53.0
Great support,6,26.833333,24.5,17.0,49.0
Love genAI features,13,27.000000,29.0,-6.0,50.0
Missing features,14,25.285714,22.5,11.0,43.0
Stable but slow,9,17.444444,9.0,-5.0,51.0
Too expensive,11,30.636364,26.0,18.0,48.0


In [101]:
#buena informacion para ana;izar mas adelante


In [102]:

surveys.to_csv("data/clean/nps_surveys_clean.csv", index=False)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [104]:
from google.colab import files
files.download("data/clean/nps_surveys_clean.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>